In [1]:
""" 
Referans Noktamız Olan Denver, Colorado'nun koordinatları: 39.74°N, 105.18°W
2021, 2022 ve 2023 yıllarına ait verileri NREL API'si üzerinden çekmek istiyoruz.
Bu kod parçası, belirtilen yıllar için verileri çekip birleştirecek ve ardından tek bir DataFrame olarak kaydedecektir.
"""

import sys # Python arama yolunu güncellemek için sys modülünü ekleyelim
import os # Dosya yollarını yönetmek için os modülünü ekleyelim
import importlib # Dinamik modül yükleme ve yeniden yükleme için importlib'i ekleyelim
import pandas as pd # Veri işleme için pandas'ı ekleyelim
from dotenv import load_dotenv # .env dosyasındaki değişkenleri yüklemek için python-dotenv kütüphanesinden load_dotenv fonksiyonunu ekleyelim


# Proje kök dizinini Python arama yolunda en üste al
project_root = os.path.abspath(os.path.join('..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root) # Proje kök dizinini arama yolunun en üstüne ekleyelim

else:
    sys.path.remove(project_root) # Eğer zaten varsa, önce kaldırıp sonra tekrar ekleyelim ki güncel versiyonun yüklendiğinden emin olalım
    sys.path.insert(0, project_root)

# Yerel modülü yükle ve cache kaynaklı eski sürümü engelle
import src.data_loader as data_loader # data_loader modülünü src klasöründen içe aktaralım
importlib.reload(data_loader) # Modülü yeniden yükleyelim ki yapılan değişiklikler yansısın
fetch_nrel_data = data_loader.fetch_nrel_data # fetch_nrel_data fonksiyonunu data_loader modülünden alalım
print("Kullanılan data_loader:", data_loader.__file__) 


# .env dosyasındaki değişkenleri yükle
load_dotenv()

# API anahtarınızı, konumu ve çekilecek yılları buradan ayarlayın
api_key = os.getenv("NREL_API_KEY") 
lat, lon = 39.74, -105.18 # Denver, CO koordinatları
years = [2021, 2022, 2023] # Çekmek istediğiniz yılları buraya ekleyin

frames = [] # Yıl bazında çekilen verileri saklamak için boş bir liste oluşturalım
for y in years: 
    print(f"{y} verisi çekiliyor...") 
    df_year = fetch_nrel_data(api_key, lat, lon, y)
    if df_year is None:
        print(f"{y} için veri çekilemedi.")
        continue

    # Yıl bazında birleştirme sonrası takip kolaylığı için etiket ekle
    df_year["Requested Year"] = y
    frames.append(df_year)
    print(f"{y} verisi başarıyla çekildi. Satır sayısı: {len(df_year)}")

# Tüm yılların verilerini birleştirelim
if frames:
    my_df = pd.concat(frames, ignore_index=True) # Yıllık verileri tek bir DataFrame'de birleştirelim
    print("Toplam veri başarıyla birleştirildi!")
    print(f"Toplam satır sayısı: {len(my_df)}")
    print(my_df.head())
    print(my_df.info())
    print(my_df.describe())

    # Veriyi CSV olarak kaydedelim ki sürekli API çağırmak zorunda kalmayalım
    raw_data_path = "../data/raw/nrel_data_2021_2023_Colorado.csv"
    my_df.to_csv(raw_data_path, index=False)
    print(f"Veri '{raw_data_path}' konumuna kaydedildi.")
else:
    my_df = None
    print("Hiçbir yıl için veri çekilemedi, lütfen API yanıtını kontrol et.")

Kullanılan data_loader: /Users/melih/Solar-Forecasting-XAI/src/data_loader.py
2021 verisi çekiliyor...
2021 verisi başarıyla çekildi. Satır sayısı: 17520
2022 verisi çekiliyor...
2022 verisi başarıyla çekildi. Satır sayısı: 17520
2023 verisi çekiliyor...
2023 verisi başarıyla çekildi. Satır sayısı: 17520
Toplam veri başarıyla birleştirildi!
Toplam satır sayısı: 52560
   Year  Month  Day  Hour  Minute  GHI  DHI  DNI  Wind Speed  Temperature  \
0  2021      1    1     0       0    0    0    0         1.6         -5.6   
1  2021      1    1     0      30    0    0    0         1.6         -5.8   
2  2021      1    1     1       0    0    0    0         1.5         -6.1   
3  2021      1    1     1      30    0    0    0         1.5         -6.2   
4  2021      1    1     2       0    0    0    0         1.4         -6.3   

   Cloud Type  Dew Point  Relative Humidity  Solar Zenith Angle  \
0           0      -10.6              67.87              163.23   
1           0      -10.6         

In [ ]:
# Datetime Indexing ile year-month-day hour:minute olarak birleştirme yapılması
my_df['datetime'] = pd.to_datetime(
    my_df[['Year','Month','Day','Hour','Minute']].rename(columns={'Year':'year',
                                                  'Month':'month',
                                                  'Day':'day',
                                                  'Hour': 'hour',
                                                  'Minute':'minute'}))
my_df.set_index('datetime', inplace=True)
print(my_df.head())

                     Year  Month  Day  Hour  Minute  GHI  DHI  DNI  \
datetime                                                             
2021-01-01 00:00:00  2021      1    1     0       0    0    0    0   
2021-01-01 00:30:00  2021      1    1     0      30    0    0    0   
2021-01-01 01:00:00  2021      1    1     1       0    0    0    0   
2021-01-01 01:30:00  2021      1    1     1      30    0    0    0   
2021-01-01 02:00:00  2021      1    1     2       0    0    0    0   

                     Wind Speed  Temperature  Cloud Type  Dew Point  \
datetime                                                              
2021-01-01 00:00:00         1.6         -5.6           0      -10.6   
2021-01-01 00:30:00         1.6         -5.8           0      -10.6   
2021-01-01 01:00:00         1.5         -6.1           7      -10.0   
2021-01-01 01:30:00         1.5         -6.2           8       -9.9   
2021-01-01 02:00:00         1.4         -6.3           7       -9.5   

           

In [13]:
# int64 ve float64 olan tüm verileri downcasting kullanarak float32 ve int32'ye çevirme
my_df.info()
for col in my_df.columns:
    if my_df[col].dtype == 'int64':
        my_df[col] = my_df[col].astype('int32')
    elif my_df[col].dtype == 'float64':
        my_df[col] = my_df[col].astype('float32')
my_df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 52560 entries, 2021-01-01 00:00:00 to 2023-12-31 23:30:00
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Year                52560 non-null  int32  
 1   Month               52560 non-null  int32  
 2   Day                 52560 non-null  int32  
 3   Hour                52560 non-null  int32  
 4   Minute              52560 non-null  int32  
 5   GHI                 52560 non-null  int32  
 6   DHI                 52560 non-null  int32  
 7   DNI                 52560 non-null  int32  
 8   Wind Speed          52560 non-null  float32
 9   Temperature         52560 non-null  float32
 10  Cloud Type          52560 non-null  int32  
 11  Dew Point           52560 non-null  float32
 12  Relative Humidity   52560 non-null  float32
 13  Solar Zenith Angle  52560 non-null  float32
 14  Requested Year      52560 non-null  int32  
dtypes: float32(5), int32(10)
memo

In [14]:
# CSV formatından Parquet formatına çevirme
import pyarrow as pa 
import pyarrow.parquet as pq

table = pa.Table.from_pandas(my_df)

pq.write_table(table, '../data/raw/nrel_data_2021_2023_Colorado.parquet')

table = pq.read_table('../data/raw/nrel_data_2021_2023_Colorado.parquet')

print(table)

pyarrow.Table
Year: int32
Month: int32
Day: int32
Hour: int32
Minute: int32
GHI: int32
DHI: int32
DNI: int32
Wind Speed: float
Temperature: float
Cloud Type: int32
Dew Point: float
Relative Humidity: float
Solar Zenith Angle: float
Requested Year: int32
datetime: timestamp[us]
----
Year: [[2021,2021,2021,2021,2021,...,2023,2023,2023,2023,2023]]
Month: [[1,1,1,1,1,...,12,12,12,12,12]]
Day: [[1,1,1,1,1,...,31,31,31,31,31]]
Hour: [[0,0,1,1,2,...,21,22,22,23,23]]
Minute: [[0,30,0,30,0,...,30,0,30,0,30]]
GHI: [[0,0,0,0,0,...,0,0,0,0,0]]
DHI: [[0,0,0,0,0,...,0,0,0,0,0]]
DNI: [[0,0,0,0,0,...,0,0,0,0,0]]
Wind Speed: [[1.6,1.6,1.5,1.5,1.4,...,0.5,0.5,0.5,0.4,0.4]]
Temperature: [[-5.6,-5.8,-6.1,-6.2,-6.3,...,-0.7,-0.7,-0.6,-0.5,-0.6]]
...
